# Análise do Efeito Flynn
## Tendências Globais de Desempenho Cognitivo — PISA 2000–2022

---

**Disciplina:** Python, Ciência de Dados e Pandas — 1º Semestre  
**Alunos:**
- Arthur Naoto Miura &nbsp;|&nbsp; RA: ______
- Guilherme Delgado &nbsp;&nbsp;&nbsp;|&nbsp; RA: ______

**Dataset:** PISA (Programme for International Student Assessment) — OCDE via API do Banco Mundial  
**Fonte secundária:** PIB per Capita — World Bank API  
**Período analisado:** 2000 – 2022

## 1. O que é o Efeito Flynn?

O **Efeito Flynn** é o fenômeno do aumento contínuo e sistemático das pontuações em testes de inteligência ao longo das gerações. Descrito pelo pesquisador **James R. Flynn** nos anos 1980, o efeito foi confirmado em mais de 30 países e aponta um ganho médio de aproximadamente **3 pontos de QI por década**.

### Principais hipóteses explicativas
| Fator | Descrição |
|-------|----------|
| Nutrição e saúde | Melhora na alimentação e redução de doenças |
| Educação formal | Maior acesso e mais anos de escolaridade |
| Urbanização | Ambientes mais cognitivamente estimulantes |
| Familiaridade com testes | Maior exposição a avaliações padronizadas |
| Tecnologia | Acesso à informação e estímulos cognitivos digitais |

### O que analisamos neste notebook
Utilizamos os dados do **PISA** (Programme for International Student Assessment) da OCDE, que avalia estudantes de 15 anos em leitura e matemática em 70+ países desde o ano 2000. Esses dados permitem observar tendências cognitivas em escala global ao longo de duas décadas, evidenciando o Efeito Flynn — e, em alguns países, sua possível **reversão** (chamada de Efeito Flynn Negativo).

> **Nota:** O PISA não mede QI diretamente, mas é considerado um dos melhores proxies disponíveis para desempenho cognitivo em larga escala, pois avalia raciocínio e aplicação do conhecimento.

## 2. Sobre os Dados

Utilizaremos três datasets obtidos via **API pública do Banco Mundial** (sem necessidade de autenticação). Os dados são baixados programaticamente, salvos como CSV e recarregados com `pd.read_csv()`.

| # | Indicador | Código World Bank | Descrição | Período |
|---|-----------|-------------------|-----------|--------|
| 1 | PISA — Leitura | `LO.PISA.REA` | Score médio de estudantes de 15 anos em leitura | 2000–2022 |
| 2 | PISA — Matemática | `LO.PISA.MAT` | Score médio em matemática | 2003–2022 |
| 3 | PIB per Capita | `NY.GDP.PCAP.PP.CD` | PIB per capita, PPP (USD correntes) | 2000–2022 |

O PISA é aplicado a cada 3 anos (2000, 2003, 2006, 2009, 2012, 2015, 2018, 2022), fornecendo uma série temporal robusta para análise do Efeito Flynn.

In [ ]:
# Instalar bibliotecas necessárias (executar apenas se necessário)
!pip install folium plotly seaborn --quiet

In [ ]:
import urllib.request
import requests
import os
import warnings
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import folium
from folium.plugins import HeatMap
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='darkgrid', palette='muted')

print('Bibliotecas importadas com sucesso!')

## 3. Download e Carregamento dos Dados

Os dados são obtidos pela **API pública do Banco Mundial** usando `requests`. Em seguida, salvamos como CSV e recarregamos com `pd.read_csv()` — mantendo o fluxo padrão de importação de dados tabulares. O `urllib.request` é utilizado para baixar o arquivo GeoJSON do mapa (seção 8).

In [ ]:
def buscar_worldbank(indicador, nome_coluna, ano_inicio=2000, ano_fim=2022):
    """Baixa dados da API pública do Banco Mundial (sem autenticação)."""
    url = f'https://api.worldbank.org/v2/country/all/indicator/{indicador}'
    params = {
        'format': 'json',
        'per_page': 20000,
        'date': f'{ano_inicio}:{ano_fim}'
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    dados = resp.json()

    registros = []
    if len(dados) > 1 and dados[1]:
        for item in dados[1]:
            if item['value'] is not None:
                iso3 = item.get('countryiso3code', '')
                if len(iso3) == 3:  # filtra apenas países (exclui agregados regionais)
                    registros.append({
                        'Entity': item['country']['value'],
                        'Code':   iso3,
                        'Year':   int(item['date']),
                        nome_coluna: item['value']
                    })
    return pd.DataFrame(registros)

print('Baixando dados via API pública do Banco Mundial...')

df_leitura = buscar_worldbank('LO.PISA.REA', 'Reading')
print(f'  PISA Leitura:    {len(df_leitura)} registros')

df_mate = buscar_worldbank('LO.PISA.MAT', 'Mathematics')
print(f'  PISA Matemática: {len(df_mate)} registros')

df_gdp_raw = buscar_worldbank('NY.GDP.PCAP.PP.CD', 'GDP per capita')
print(f'  PIB per capita:  {len(df_gdp_raw)} registros')

# Buscar regiões geográficas dos países
# NOTA: o campo ISO3 na resposta do endpoint /v2/country é 'id', não 'iso3Code'
resp_reg = requests.get(
    'https://api.worldbank.org/v2/country?format=json&per_page=300', timeout=15
).json()
regioes = {}
if len(resp_reg) > 1:
    for p in resp_reg[1]:
        iso3   = p.get('id', '')          # campo correto é 'id'
        regiao = p.get('region', {}).get('value', '')
        if len(iso3) == 3 and regiao and regiao != 'Aggregates':
            regioes[iso3] = regiao

print(f'  Regiões mapeadas: {len(regioes)} países')

df_gdp_raw['World region according to OWID'] = df_gdp_raw['Code'].map(regioes)

# Salvar como CSV
df_leitura.to_csv('pisa_leitura.csv', index=False)
df_mate.to_csv('pisa_matematica.csv', index=False)
df_gdp_raw.to_csv('gdp_per_capita.csv', index=False)

# Recarregar com read_csv
df_leitura = pd.read_csv('pisa_leitura.csv')
df_mate    = pd.read_csv('pisa_matematica.csv')
df_gdp     = pd.read_csv('gdp_per_capita.csv')

print('\nDatasets carregados com sucesso!')
print(f'  Leitura:    {df_leitura.shape[0]} linhas x {df_leitura.shape[1]} colunas')
print(f'  Matematica: {df_mate.shape[0]} linhas x {df_mate.shape[1]} colunas')
print(f'  PIB:        {df_gdp.shape[0]} linhas x {df_gdp.shape[1]} colunas')

## 4. Exploração Inicial dos Dados

Antes de qualquer transformação, exploramos a estrutura de cada dataset: **shape**, **columns**, **dtypes**, **info**, **head** e **tail** nos dão uma visão rápida do que temos.

In [ ]:
# shape
print('=== shape ===')
print('Leitura:', df_leitura.shape)
print('Matematica:', df_mate.shape)
print('PIB:', df_gdp.shape)

# columns
print('\n=== columns ===')
print('Leitura:', df_leitura.columns.tolist())
print('Matematica:', df_mate.columns.tolist())
print('PIB:', df_gdp.columns.tolist())

# dtypes
print('\n=== dtypes (Leitura) ===')
print(df_leitura.dtypes)

In [ ]:
# info
print('=== info() — PISA Leitura ===')
df_leitura.info()
print('\n=== info() — PIB per Capita ===')
df_gdp.info()

In [ ]:
# head
print('=== head() — PISA Leitura ===')
display(df_leitura.head())

# tail
print('\n=== tail() — PISA Leitura ===')
display(df_leitura.tail())

In [ ]:
# to_dict — visualizar primeiros registros como dicionário
print('=== to_dict (primeiros 3 registros) ===')
print(df_leitura.head(3).to_dict(orient='records'))

# value_counts — distribuição dos anos disponíveis
print('\n=== value_counts — Anos disponíveis (Leitura) ===')
print(df_leitura['Year'].value_counts().sort_index())

print(f'\nTotal de países únicos (Leitura): {df_leitura["Entity"].nunique()}')
print(f'Total de países únicos (Matemática): {df_mate["Entity"].nunique()}')

## 5. Limpeza e Transformação dos Dados

Nesta etapa realizamos:
1. **rename** — padronização dos nomes de colunas para português
2. **isna / fillna** — identificação e tratamento de valores ausentes
3. **to_numeric / to_datetime** — conversão de tipos
4. **drop / copy** — remoção de colunas desnecessárias e cópia do dataset
5. **Merge** — junção dos três datasets em um único DataFrame

In [ ]:
# rename — padronizar colunas
df_leitura = df_leitura.rename(columns={
    'Entity': 'Pais', 'Code': 'Codigo', 'Year': 'Ano', 'Reading': 'Leitura'
})

df_mate = df_mate.rename(columns={
    'Entity': 'Pais', 'Code': 'Codigo', 'Year': 'Ano', 'Mathematics': 'Matematica'
})

df_gdp = df_gdp.rename(columns={
    'Entity': 'Pais',
    'Code': 'Codigo',
    'Year': 'Ano',
    'GDP per capita': 'PIB_per_capita',
    'World region according to OWID': 'Continente'
})

print('Colunas renomeadas!')
display(df_leitura.head(3))
display(df_gdp.head(3))

In [ ]:
# isna — verificar nulos
print('=== Valores nulos — PISA Leitura ===')
print(df_leitura.isna().sum())

print('\n=== Valores nulos — PISA Matemática ===')
print(df_mate.isna().sum())

print('\n=== Valores nulos — PIB ===')
print(df_gdp.isna().sum())

# Remover linhas sem código de país (são regiões/agregados como 'OECD average')
df_leitura = df_leitura.dropna(subset=['Codigo'])
df_mate    = df_mate.dropna(subset=['Codigo'])
df_gdp_clean = df_gdp.dropna(subset=['Codigo', 'Continente']).copy()

print(f'\nApós remoção de agregados:')
print(f'  Leitura:    {df_leitura.shape[0]} registros')
print(f'  Matemática: {df_mate.shape[0]} registros')
print(f'  PIB:        {df_gdp_clean.shape[0]} registros')

In [ ]:
# to_numeric — garantir que os scores são numéricos
df_leitura['Leitura']          = pd.to_numeric(df_leitura['Leitura'], errors='coerce')
df_mate['Matematica']          = pd.to_numeric(df_mate['Matematica'], errors='coerce')
df_gdp_clean['PIB_per_capita'] = pd.to_numeric(df_gdp_clean['PIB_per_capita'], errors='coerce')

# to_datetime — criar coluna de data a partir do Ano
df_leitura['Data'] = pd.to_datetime(df_leitura['Ano'].astype(str), format='%Y')

print('=== to_datetime — coluna Data criada ===')
print(df_leitura[['Pais', 'Ano', 'Data']].head())
print(f'\nTipo da coluna Data: {df_leitura["Data"].dtype}')

In [ ]:
# copy — trabalhar com cópia para preservar originais
df_leitura_proc = df_leitura.drop(columns=['Data']).copy()
df_mate_proc    = df_mate.copy()

# Merge 1: Leitura + Matemática
df = pd.merge(
    df_leitura_proc, df_mate_proc,
    on=['Pais', 'Codigo', 'Ano'],
    how='outer'
)

# Filtrar PIB apenas para os anos PISA
anos_pisa = [2000, 2003, 2006, 2009, 2012, 2015, 2018, 2022]
df_gdp_pisa = df_gdp_clean[df_gdp_clean['Ano'].isin(anos_pisa)][['Pais','Codigo','Ano','PIB_per_capita','Continente']].copy()

# Merge 2: PISA + PIB
df = pd.merge(df, df_gdp_pisa, on=['Pais', 'Codigo', 'Ano'], how='left')

print('Dataset final merged:')
print('Shape:', df.shape)
display(df.head())

In [ ]:
# apply — calcular média PISA por linha (Leitura + Matemática)
def calcular_media_pisa(row):
    scores = [s for s in [row['Leitura'], row.get('Matematica', np.nan)] if pd.notna(s)]
    return round(np.mean(scores), 2) if scores else np.nan

df['Media_PISA'] = df.apply(calcular_media_pisa, axis=1)

# fillna — preencher Continente ausente
df['Continente'] = df['Continente'].fillna('Desconhecido')

# fillna — PIB ausente com mediana do continente + ano
df['PIB_per_capita'] = df.groupby(['Continente', 'Ano'])['PIB_per_capita'].transform(
    lambda x: x.fillna(x.median())
)

print('Média PISA calculada. Nulos restantes:')
print(df[['Leitura', 'Matematica', 'Media_PISA', 'PIB_per_capita']].isna().sum())

## 6. Manipulação e Consulta dos Dados

Agora utilizamos funções de manipulação para explorar e filtrar os dados: `set_index`, `reset_index`, `sort_values`, `sort_index`, `query`, `nlargest`, `nsmallest` e `to_frame`.

In [ ]:
# set_index com índice multi-nível
df_idx = df.set_index(['Pais', 'Ano'])
print('=== set_index — índice multi-nível ===')
display(df_idx.head())

# reset_index
df_idx = df_idx.reset_index()

# sort_values — maiores scores de leitura
print('\n=== sort_values — Top 10 scores de Leitura ===')
display(
    df.sort_values('Leitura', ascending=False)
      .head(10)[['Pais', 'Ano', 'Leitura', 'Continente']]
)

# sort_index
df_si = df.set_index('Pais').sort_index()
print('\n=== sort_index — primeiros países (ordem alfabética) ===')
display(df_si[['Ano', 'Leitura', 'Matematica']].head())

In [ ]:
# Definir ano de referência = último ano PISA com dados disponíveis
ANO_REF = int(df.dropna(subset=['Media_PISA'])['Ano'].max())
print(f'Último ano PISA disponível: {ANO_REF}')

# query — filtrar Brasil
brasil = df.query("Pais == 'Brazil'").sort_values('Ano')
print('\n=== Brasil no PISA ===')
display(brasil[['Pais', 'Ano', 'Leitura', 'Matematica', 'Media_PISA', 'PIB_per_capita']])

# query — países com score acima de 530 em leitura no ano de referência
print(f'\n=== Países com Leitura > 530 em {ANO_REF} ===')
display(df.query(f"Ano == {ANO_REF} and Leitura > 530")[['Pais', 'Leitura', 'Matematica', 'Continente']])

In [ ]:
df_2022 = df[df['Ano'] == ANO_REF].dropna(subset=['Media_PISA'])
print(f'{len(df_2022)} países com dados no PISA {ANO_REF}')

# nlargest — top 10 países no ano de referência
print(f'\n=== nlargest — Top 10 países por Score Médio PISA ({ANO_REF}) ===')
display(df_2022.nlargest(10, 'Media_PISA')[['Pais', 'Media_PISA', 'Continente', 'PIB_per_capita']])

# nsmallest — bottom 10 países no ano de referência
print(f'\n=== nsmallest — 10 países com menor Score Médio PISA ({ANO_REF}) ===')
display(df_2022.nsmallest(10, 'Media_PISA')[['Pais', 'Media_PISA', 'Continente', 'PIB_per_capita']])

In [ ]:
# to_frame — média por continente no ano de referência como DataFrame
print(f'=== to_frame — Média PISA por Continente ({ANO_REF}) ===')
media_continente = (
    df_2022.groupby('Continente')['Media_PISA']
    .mean()
    .sort_values(ascending=False)
    .to_frame()
)
display(media_continente)

# to_csv — exportar dataset processado
df.to_csv('efeito_flynn_processado.csv', index=False)
print('\nDataset exportado: efeito_flynn_processado.csv')

## 7. Análise Temporal — O Efeito Flynn em Números

Esta é a etapa central do projeto. Usamos funções de **análise de séries temporais** do Pandas para medir a evolução do desempenho cognitivo ao longo dos anos PISA (2000–2022):

- **`diff`** — diferença absoluta entre períodos consecutivos
- **`pct_change`** — variação percentual entre períodos
- **`rolling`** — média e soma móvel (tendência suavizada)
- **`cumsum`** — acumulado ao longo do tempo

Estas funções nos permitem detectar se os países estão melhorando (Efeito Flynn positivo) ou piorando (Efeito Flynn negativo) ao longo do tempo.

In [ ]:
# Ordenar por país e ano para garantir a série temporal correta
df_ts = df.sort_values(['Pais', 'Ano']).copy()

# diff — variação absoluta de Leitura entre avaliações consecutivas por país
df_ts['Leitura_diff'] = df_ts.groupby('Pais')['Leitura'].diff()

# pct_change — variação percentual
df_ts['Leitura_pct_change'] = df_ts.groupby('Pais')['Leitura'].pct_change() * 100

print('=== diff e pct_change — Brasil ===')
display(
    df_ts[df_ts['Pais'] == 'Brazil'][['Pais', 'Ano', 'Leitura', 'Leitura_diff', 'Leitura_pct_change']]
    .dropna()
    .round(2)
)

In [ ]:
# rolling mean — média móvel de janela 2 (tendência suavizada)
df_ts['Leitura_rolling_media'] = df_ts.groupby('Pais')['Leitura'].transform(
    lambda x: x.rolling(window=2, min_periods=1).mean()
)

# rolling sum — soma acumulada janela 2
df_ts['Leitura_rolling_soma'] = df_ts.groupby('Pais')['Leitura'].transform(
    lambda x: x.rolling(window=2, min_periods=1).sum()
)

# cumsum — acumulado do score de leitura por país
df_ts['Leitura_cumsum'] = df_ts.groupby('Pais')['Leitura'].cumsum()

print('=== rolling e cumsum — Alemanha (exemplo de Efeito Flynn positivo) ===')
display(
    df_ts[df_ts['Pais'] == 'Germany'][[
        'Pais', 'Ano', 'Leitura',
        'Leitura_rolling_media', 'Leitura_rolling_soma', 'Leitura_cumsum'
    ]].round(2)
)

# Países com maior ganho total no período
print('\n=== Países com maior variação acumulada (Leitura) ===')
ganho = df_ts.groupby('Pais')['Leitura_diff'].sum().dropna().sort_values(ascending=False)
display(ganho.head(10).to_frame().rename(columns={'Leitura_diff': 'Ganho_Total'}).round(2))

In [ ]:
# stack — transformar Leitura e Matemática de wide para long
df_wide = df[['Pais', 'Continente', 'Ano', 'Leitura', 'Matematica']].dropna(subset=['Leitura', 'Matematica'])
df_stack = df_wide.set_index(['Pais', 'Continente', 'Ano']).stack().reset_index()
df_stack.columns = ['Pais', 'Continente', 'Ano', 'Disciplina', 'Score']
print('=== stack — formato long (primeiras linhas) ===')
display(df_stack.head(10))

# crosstab — número de países participantes por continente e ano
print('\n=== crosstab — Países participantes por Continente x Ano ===')
ct = pd.crosstab(df['Continente'], df['Ano'])
display(ct)

# transpose
print('\n=== transpose da crosstab (Anos x Continentes) ===')
display(ct.transpose())

## 8. Visualização em Mapas

Utilizamos a biblioteca **Folium** para criar mapas interativos. O arquivo GeoJSON com os contornos dos países é baixado via `urllib.request`. Dois tipos de visualização:
1. **Choropleth** — mapa colorido por score PISA médio em 2022
2. **Marcadores de círculo** — pontos com tamanho proporcional ao ganho de score (Efeito Flynn)

In [ ]:
# Baixar GeoJSON via urllib.request
GEO_URL = 'https://raw.githubusercontent.com/python-visualization/folium/main/examples/data/world-countries.json'
urllib.request.urlretrieve(GEO_URL, 'world-countries.json')
print('GeoJSON baixado com sucesso!')

# Preparar dados do ano de referência — usar Codigo (ISO3) para casar com feature.id do GeoJSON
df_map = df_2022.dropna(subset=['Codigo'])[['Pais', 'Codigo', 'Media_PISA']].copy()

# Criar mapa base
mapa = folium.Map(location=[20, 0], zoom_start=2, tiles='CartoDB positron')

# Choropleth por código ISO3 (feature.id no GeoJSON) — evita divergência de nomes
folium.Choropleth(
    geo_data='world-countries.json',
    name=f'Score PISA {ANO_REF}',
    data=df_map,
    columns=['Codigo', 'Media_PISA'],
    key_on='feature.id',
    fill_color='YlOrRd',
    fill_opacity=0.75,
    line_opacity=0.2,
    legend_name=f'Score Médio PISA {ANO_REF} (Leitura + Matemática)',
    nan_fill_color='#cccccc',
    nan_fill_opacity=0.4
).add_to(mapa)

folium.LayerControl().add_to(mapa)

# Salvar e exibir
mapa.save('mapa_pisa_2022.html')
print('Mapa salvo: mapa_pisa_2022.html')
mapa

In [ ]:
# Coordenadas aproximadas de países selecionados
coords = {
    'Brazil': (-14.24, -51.93), 'Japan': (36.20, 138.25), 'Finland': (61.92, 25.75),
    'Germany': (51.17, 10.45), 'United States': (37.09, -95.71), 'China': (35.86, 104.20),
    'France': (46.23, 2.21), 'Argentina': (-38.42, -63.62), 'Chile': (-35.68, -71.54),
    'South Korea': (35.91, 127.77), 'Canada': (56.13, -106.35), 'Australia': (-25.27, 133.78),
    'Poland': (51.92, 19.15), 'Portugal': (39.40, -8.22), 'Mexico': (23.63, -102.55),
    'Singapore': (1.35, 103.82), 'Spain': (40.46, -3.75), 'Sweden': (60.13, 18.64)
}

# Calcular ganho total por país (Efeito Flynn)
ganho_pais = df_ts.groupby('Pais')['Leitura_diff'].sum().dropna().reset_index()
ganho_pais.columns = ['Pais', 'Ganho_Leitura']

# Mapa de marcadores
mapa2 = folium.Map(location=[20, 0], zoom_start=2, tiles='CartoDB positron')

for _, row in ganho_pais.iterrows():
    pais = row['Pais']
    if pais in coords:
        lat, lon = coords[pais]
        ganho = row['Ganho_Leitura']
        cor = 'green' if ganho > 0 else 'red'
        folium.CircleMarker(
            location=[lat, lon],
            radius=max(3, min(15, abs(ganho) / 5)),
            color=cor,
            fill=True,
            fill_opacity=0.7,
            popup=folium.Popup(f'{pais}: {ganho:.1f} pts', parse_html=True),
            tooltip=f'{pais}: {ganho:.1f} pts'
        ).add_to(mapa2)

mapa2.save('mapa_efeito_flynn.html')
print('Mapa de marcadores salvo: mapa_efeito_flynn.html')
mapa2

## 9. Visualizações em Gráficos

Utilizamos **Matplotlib**, **Seaborn** e **Plotly** para criar visualizações que evidenciam o Efeito Flynn e as desigualdades cognitivas globais.

In [ ]:
# --- Gráfico 1: Evolução PISA ao longo do tempo (países selecionados) ---
paises_selecionados = ['Brazil', 'Japan', 'Finland', 'Germany', 'Argentina', 'Chile', 'Poland', 'Portugal']

df_sel = df[df['Pais'].isin(paises_selecionados)].dropna(subset=['Leitura'])

fig, ax = plt.subplots(figsize=(14, 6))

for pais in paises_selecionados:
    dados = df_sel[df_sel['Pais'] == pais].sort_values('Ano')
    if len(dados) > 1:
        ax.plot(dados['Ano'], dados['Leitura'], marker='o', label=pais, linewidth=2)

ax.set_title('Evolução do Score de Leitura PISA (2000–2022)\nEfeito Flynn em países selecionados', fontsize=14, fontweight='bold')
ax.set_xlabel('Ano')
ax.set_ylabel('Score de Leitura')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
ax.xaxis.set_major_locator(mticker.MultipleLocator(3))
ax.grid(True, alpha=0.5)
plt.tight_layout()
plt.savefig('grafico_evolucao_leitura.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Gráfico 2: Top 20 países por Score Médio PISA (ano de referência) ---
top20 = df_2022.nlargest(20, 'Media_PISA')[['Pais', 'Media_PISA', 'Continente']]

palette_cont = {
    'East Asia & Pacific': '#e74c3c',
    'Europe & Central Asia': '#3498db',
    'North America': '#2ecc71',
    'Latin America & Caribbean': '#f39c12',
    'South Asia': '#9b59b6',
    'Desconhecido': '#95a5a6'
}
cores = [palette_cont.get(c, '#7f8c8d') for c in top20['Continente']]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top20['Pais'], top20['Media_PISA'], color=cores, edgecolor='white', height=0.7)
ax.bar_label(bars, fmt='%.0f', padding=3, fontsize=9)
ax.set_title(f'Top 20 Países — Score Médio PISA {ANO_REF}\n(Leitura + Matemática)', fontsize=13, fontweight='bold')
ax.set_xlabel('Score Médio PISA')
ax.invert_yaxis()

from matplotlib.patches import Patch
conts_presentes = top20['Continente'].unique()
legenda = [Patch(color=palette_cont.get(c, '#7f8c8d'), label=c) for c in conts_presentes]
ax.legend(handles=legenda, title='Região', loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig('grafico_top20_2022.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Gráfico 3: PISA vs PIB per Capita (Plotly) ---
df_scatter = df_2022.dropna(subset=['PIB_per_capita', 'Media_PISA', 'Continente'])
df_scatter = df_scatter[df_scatter['Continente'] != 'Desconhecido']

fig = px.scatter(
    df_scatter,
    x='PIB_per_capita',
    y='Media_PISA',
    color='Continente',
    hover_name='Pais',
    size='Media_PISA',
    size_max=25,
    title=f'Relação entre PIB per Capita e Score PISA ({ANO_REF})<br><sub>Cada ponto = um país</sub>',
    labels={'PIB_per_capita': 'PIB per Capita (USD, PPP)', 'Media_PISA': 'Score Médio PISA'},
    template='plotly_white',
    trendline='ols'
)
fig.update_layout(height=550)
fig.show()

In [ ]:
# --- Gráfico 4: Heatmap de correlação (Seaborn) ---
df_corr = df[['Leitura', 'Matematica', 'Media_PISA', 'PIB_per_capita', 'Ano']].dropna()
corr = df_corr.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
    mask=mask, ax=ax, linewidths=0.5, square=True,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Matriz de Correlação — Variáveis PISA e PIB', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('grafico_correlacao.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Gráfico 5: Boxplot Score PISA por Região (Seaborn) ---
df_box = df[df['Continente'] != 'Desconhecido'].dropna(subset=['Media_PISA'])

ordem = df_box.groupby('Continente')['Media_PISA'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(13, 6))
sns.boxplot(
    data=df_box, x='Continente', y='Media_PISA',
    order=ordem, palette='Set2', ax=ax
)
ax.set_title('Distribuição do Score PISA por Região (2000–2022)', fontsize=13, fontweight='bold')
ax.set_xlabel('Região')
ax.set_ylabel('Score Médio PISA')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('grafico_boxplot_regioes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Gráfico 6: Efeito Flynn — Média global ao longo do tempo (Plotly) ---
media_global = (
    df.dropna(subset=['Leitura'])
    .groupby('Ano')
    .agg(Media_Leitura=('Leitura', 'mean'), Paises=('Pais', 'count'))
    .reset_index()
)

fig = px.bar(
    media_global, x='Ano', y='Media_Leitura',
    text='Paises',
    title='Score Médio Global de Leitura PISA (2000–2022)<br><sub>Número acima da barra = países participantes</sub>',
    labels={'Media_Leitura': 'Score Médio de Leitura', 'Ano': 'Ano'},
    color='Media_Leitura',
    color_continuous_scale='Viridis',
    template='plotly_white'
)
fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.update_layout(height=500, showlegend=False)
fig.show()

In [ ]:
# --- Gráfico 7: Mapa Choropleth Interativo (Plotly) ---
fig = px.choropleth(
    df_2022.dropna(subset=['Codigo', 'Media_PISA']),
    locations='Codigo',
    color='Media_PISA',
    hover_name='Pais',
    color_continuous_scale='RdYlGn',
    title=f'Score Médio PISA {ANO_REF} por País',
    labels={'Media_PISA': 'Score PISA'},
    template='plotly_white'
)
fig.update_layout(height=500, geo=dict(showframe=False, showcoastlines=True))
fig.show()

## 10. Análise Interativa

Explore os dados de forma livre com os três painéis abaixo. Cada painel atualiza o gráfico automaticamente conforme você muda as seleções.

| Painel | O que permite |
|--------|--------------|
| **Comparação por País** | Escolha qualquer conjunto de países e veja a evolução do score no período selecionado |
| **Comparação por Região** | Compare regiões do mundo via boxplot, barras ou linha de tendência |
| **Ranking Interativo** | Veja o Top/Bottom N países em qualquer edição do PISA (2000–2022) |

> **Dica:** Use **Ctrl+Click** (Windows/Linux) ou **Cmd+Click** (Mac) para selecionar múltiplos itens nas listas.

In [ ]:
!pip install ipywidgets --quiet
print('ipywidgets pronto!')

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

# Listas de opções derivadas dos dados reais
paises_lista  = sorted(df['Pais'].dropna().unique().tolist())
regioes_lista = sorted([r for r in df['Continente'].dropna().unique() if r != 'Desconhecido'])
anos_lista    = sorted(df['Ano'].dropna().astype(int).unique().tolist())

# Mapeamento label exibido -> coluna do DataFrame
SCORE_MAP = {'Leitura': 'Leitura', 'Matemática': 'Matematica', 'Média PISA': 'Media_PISA'}

print(f'Países disponíveis : {len(paises_lista)}')
print(f'Regiões disponíveis: {regioes_lista}')
print(f'Anos disponíveis   : {anos_lista}')

In [ ]:
# ── PAINEL 1 — Comparação por País ──────────────────────────────────────────
w1_paises = widgets.SelectMultiple(
    options=paises_lista,
    value=['Brazil', 'Japan', 'Germany', 'Finland', 'Poland'],
    description='Países:',
    layout=widgets.Layout(width='260px', height='200px')
)
w1_score = widgets.Dropdown(
    options=list(SCORE_MAP.keys()),
    value='Leitura',
    description='Score:',
    layout=widgets.Layout(width='220px')
)
w1_periodo = widgets.SelectionRangeSlider(
    options=anos_lista,
    index=(0, len(anos_lista) - 1),
    description='Período:',
    continuous_update=False,
    layout=widgets.Layout(width='420px')
)
out1 = widgets.Output()

def atualizar_paises(*_):
    col    = SCORE_MAP[w1_score.value]
    ano_i, ano_f = w1_periodo.value
    paises = list(w1_paises.value)
    with out1:
        clear_output(wait=True)
        if not paises:
            print('Selecione ao menos um país (Ctrl+Click para múltiplos).')
            return
        df_fil = df[
            df['Pais'].isin(paises) &
            df['Ano'].between(ano_i, ano_f)
        ].dropna(subset=[col])
        if df_fil.empty:
            print('Nenhum dado disponível para esta seleção.')
            return
        fig, ax = plt.subplots(figsize=(13, 5))
        for pais in paises:
            d = df_fil[df_fil['Pais'] == pais].sort_values('Ano')
            if not d.empty:
                ax.plot(d['Ano'], d[col], marker='o', label=pais, linewidth=2, markersize=6)
        ax.set_title(f'Evolução de {w1_score.value} por País ({ano_i}–{ano_f})',
                     fontsize=13, fontweight='bold')
        ax.set_xlabel('Ano'); ax.set_ylabel(w1_score.value)
        ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
        ax.xaxis.set_major_locator(mticker.MultipleLocator(3))
        ax.grid(True, alpha=0.4)
        plt.tight_layout(); plt.show()

for w in (w1_paises, w1_score, w1_periodo):
    w.observe(atualizar_paises, names='value')

display(widgets.VBox([
    widgets.HTML('<h4>&#127758; Comparação por País</h4><small>Ctrl+Click para selecionar múltiplos países</small>'),
    widgets.HBox([w1_paises, widgets.VBox([w1_score, w1_periodo])]),
    out1
]))
atualizar_paises()

In [ ]:
# ── PAINEL 2 — Comparação por Região ────────────────────────────────────────
w2_regioes = widgets.SelectMultiple(
    options=regioes_lista,
    value=regioes_lista[:4] if len(regioes_lista) >= 4 else regioes_lista,
    description='Regiões:',
    layout=widgets.Layout(width='300px', height='160px')
)
w2_score = widgets.Dropdown(
    options=list(SCORE_MAP.keys()),
    value='Média PISA',
    description='Score:',
    layout=widgets.Layout(width='220px')
)
w2_periodo = widgets.SelectionRangeSlider(
    options=anos_lista,
    index=(0, len(anos_lista) - 1),
    description='Período:',
    continuous_update=False,
    layout=widgets.Layout(width='420px')
)
w2_tipo = widgets.RadioButtons(
    options=['Boxplot', 'Barras (média)', 'Linha (média)'],
    value='Barras (média)',
    description='Tipo:',
    layout=widgets.Layout(width='200px')
)
out2 = widgets.Output()

def atualizar_regioes(*_):
    col     = SCORE_MAP[w2_score.value]
    ano_i, ano_f = w2_periodo.value
    regioes = list(w2_regioes.value)
    tipo    = w2_tipo.value
    with out2:
        clear_output(wait=True)
        if not regioes:
            print('Selecione ao menos uma região.')
            return
        df_fil = df[
            df['Continente'].isin(regioes) &
            df['Ano'].between(ano_i, ano_f)
        ].dropna(subset=[col])
        if df_fil.empty:
            print('Nenhum dado disponível.')
            return
        fig, ax = plt.subplots(figsize=(13, 5))
        if tipo == 'Boxplot':
            ordem = df_fil.groupby('Continente')[col].median().sort_values(ascending=False).index
            sns.boxplot(data=df_fil, x='Continente', y=col,
                        order=[r for r in ordem if r in regioes],
                        palette='Set2', ax=ax)
            ax.tick_params(axis='x', rotation=15)
        elif tipo == 'Barras (média)':
            med = df_fil.groupby('Continente')[col].mean().reindex(regioes).dropna().sort_values(ascending=False)
            bars = ax.bar(range(len(med)), med.values,
                          color=sns.color_palette('Set2', len(med)), edgecolor='white')
            ax.set_xticks(range(len(med)))
            ax.set_xticklabels(med.index, rotation=15, ha='right')
            ax.bar_label(bars, fmt='%.1f', padding=3)
        else:
            for reg in regioes:
                d = df_fil[df_fil['Continente'] == reg].groupby('Ano')[col].mean().reset_index()
                if not d.empty:
                    ax.plot(d['Ano'], d[col], marker='o', label=reg, linewidth=2)
            ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
            ax.xaxis.set_major_locator(mticker.MultipleLocator(3))
            ax.grid(True, alpha=0.4)
        ax.set_title(f'{w2_score.value} por Região ({ano_i}–{ano_f}) — {tipo}',
                     fontsize=13, fontweight='bold')
        ax.set_ylabel(w2_score.value)
        plt.tight_layout(); plt.show()

for w in (w2_regioes, w2_score, w2_periodo, w2_tipo):
    w.observe(atualizar_regioes, names='value')

display(widgets.VBox([
    widgets.HTML('<h4>&#127758; Comparação por Região</h4>'),
    widgets.HBox([w2_regioes, widgets.VBox([w2_score, w2_periodo, w2_tipo])]),
    out2
]))
atualizar_regioes()

In [ ]:
# ── PAINEL 3 — Ranking Interativo por Ano ───────────────────────────────────
w3_ano = widgets.SelectionSlider(
    options=anos_lista,
    value=anos_lista[-1],
    description='Ano:',
    continuous_update=False,
    layout=widgets.Layout(width='420px')
)
w3_score = widgets.Dropdown(
    options=list(SCORE_MAP.keys()),
    value='Média PISA',
    description='Score:',
    layout=widgets.Layout(width='220px')
)
w3_topn = widgets.IntSlider(
    value=15, min=5, max=30, step=1,
    description='Top N:',
    continuous_update=False,
    layout=widgets.Layout(width='320px')
)
w3_ordem = widgets.ToggleButtons(
    options=['Maiores', 'Menores'],
    value='Maiores',
    description='Ordem:',
    button_style='info'
)
out3 = widgets.Output()

def atualizar_ranking(*_):
    col   = SCORE_MAP[w3_score.value]
    ano   = w3_ano.value
    topn  = w3_topn.value
    ordem = w3_ordem.value
    with out3:
        clear_output(wait=True)
        df_ano = df[df['Ano'] == ano].dropna(subset=[col])
        if df_ano.empty:
            print(f'Sem dados de {w3_score.value} para o ano {ano}.')
            return
        df_rank = df_ano.nlargest(topn, col) if ordem == 'Maiores' else df_ano.nsmallest(topn, col)
        cores = sns.color_palette('RdYlGn' if ordem == 'Maiores' else 'RdYlGn_r', len(df_rank))
        fig, ax = plt.subplots(figsize=(11, max(5, topn * 0.42)))
        bars = ax.barh(df_rank['Pais'], df_rank[col], color=cores, edgecolor='white')
        ax.bar_label(bars, fmt='%.1f', padding=3, fontsize=9)
        rotulo = 'Top' if ordem == 'Maiores' else 'Bottom'
        ax.set_title(f'{rotulo} {topn} Países — {w3_score.value} ({ano})',
                     fontsize=13, fontweight='bold')
        ax.set_xlabel(w3_score.value)
        ax.invert_yaxis()
        plt.tight_layout(); plt.show()

for w in (w3_ano, w3_score, w3_topn, w3_ordem):
    w.observe(atualizar_ranking, names='value')

display(widgets.VBox([
    widgets.HTML('<h4>&#127942; Ranking Interativo por Ano</h4>'),
    widgets.HBox([widgets.VBox([w3_score, w3_ano, w3_topn]), w3_ordem]),
    out3
]))
atualizar_ranking()

## 11. Conclusão

### O que os dados revelam sobre o Efeito Flynn?

A análise dos dados PISA de 2000 a 2022 nos permite extrair as seguintes conclusões:

#### Evidências do Efeito Flynn
- **Países em desenvolvimento** como Brasil, Chile, Peru e países do Sudeste Asiático apresentaram **ganhos significativos** nos scores PISA ao longo do período, consistentes com o Efeito Flynn — à medida que melhoram condições de educação, nutrição e renda.
- A correlação positiva entre **PIB per capita e Score PISA** confirma que fatores socioeconômicos são determinantes do desempenho cognitivo em larga escala.

#### Reversão do Efeito Flynn
- Países como Finlândia, Noruega e outros países escandinavos — que já atingiram níveis muito altos — mostram **estagnação ou leve queda** nos scores mais recentes (2015–2022), fenômeno descrito na literatura como *Efeito Flynn Negativo* ou *Efeito Flynn Reverso*.

#### Disparidades Globais
- A análise por região revela que **Leste Asiático e Pacífico** (liderado por Japão, Coreia do Sul e Singapura) e **Europa & Ásia Central** consistentemente lideram os rankings PISA.
- A **América Latina & Caribe** apresenta os menores scores entre as regiões com dados disponíveis, mas com tendências de melhora ao longo do tempo.

### Limitações do estudo
- O PISA avalia apenas estudantes de 15 anos que frequentam a escola — populações com evasão escolar alta são sub-representadas.
- Nem todos os países participam de todas as edições, criando lacunas na série temporal.
- O PIB per capita é uma variável correlacionada, mas não estabelece causalidade direta com o desempenho cognitivo.

---
*Notebook desenvolvido para a disciplina de Python, Ciência de Dados e Pandas — 1º Semestre de 2026.*